# Stage 11 — Collective Validation Sweep (SageMaker)

Closes the project with a **systematic** result rather than a single anecdote. Three components:

1. **Baseline-gate positive/negative control** — prove the pLDDT≥70 gate discriminates a foldable wild-type chassis from a collapsed one.
2. **Multi-seed structural sweep** — run the full 11a→11d pipeline on several baseline-qualified wild-type seeds (default weights, fixed search config) and report the per-seed top-3 structural pass rate.
3. **Seed-distance characterization** — relate each seed's embedding distance to the Enterobacter centroid against its predicted target-host probability, turning the known `target_probability` ceiling into a quantified finding.

All seeds target **Enterobacter**; everything except the seed is held constant. Tested on `ml.g5.2xlarge` (A10G, 24 GB).

## 1 — Environment setup (unzip, install, patch fair-esm)

In [1]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path

HOME = Path('/home/sagemaker-user')
ZIP  = HOME / 'phageforge_stage11.zip'
ROOT = HOME / 'phageforge_clean'

# --- 1a. Unzip if needed --------------------------------------------------
if not (ROOT / 'pyproject.toml').exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    assert ZIP.exists(), f'Upload the zip to {ZIP} first.'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)

# --- 1b. Repair Windows-style backslash paths ----------------------------
# Windows-created zips dump files as `phageforge\__init__.py` on Linux extraction.
# Rebuild the directory tree.
for p in list(ROOT.iterdir()):
    if '\\' not in p.name:
        continue
    if p.name.endswith('\\'):
        p.unlink(missing_ok=True); continue
    target = ROOT.joinpath(*p.name.split('\\'))
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(p), str(target))

# Move stray data assets to expected locations
for src, dst in {
    'rbp_dataset_eskapee_strict.csv': 'data/processed/rbp_dataset_eskapee_strict.csv',
    'esm2_embeddings.pt':             'data/processed/strict/esm2_embeddings.pt',
    'esm2_embeddings_index.csv':      'data/processed/strict/esm2_embeddings_index.csv',
    'model.joblib':                   'results/broad/linear_probe/seed_42/model.joblib',
    'label_classes.json':             'results/broad/linear_probe/seed_42/label_classes.json',
}.items():
    s, d = ROOT / src, ROOT / dst
    if s.exists() and not d.exists():
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(s), str(d))
shutil.rmtree(ROOT / 'phageforge.egg-info', ignore_errors=True)
for pc in ROOT.rglob('__pycache__'):
    shutil.rmtree(pc, ignore_errors=True)

# --- 1c. Set CWD and env -------------------------------------------------
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('cwd:', os.getcwd())

# --- 1d. Install the phageforge package + runtime deps ------------------
PY = sys.executable
subprocess.run([PY, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-q',
                'fair-esm', '-U', 'biotite', 'torch-geometric'], check=True)

# --- 1e. Patch the installed fair-esm library (cannot ship this in our repo) -
import esm as _esm
ESM_UTIL = Path(_esm.__file__).parent / 'inverse_folding' / 'util.py'
if ESM_UTIL.exists():
    txt = ESM_UTIL.read_text()
    # (i) Biotite API rename: filter_backbone -> filter_peptide_backbone
    txt = txt.replace(
        'from biotite.structure import filter_backbone',
        'from biotite.structure import filter_peptide_backbone as filter_backbone',
    )
    # (ii) GPU->numpy safety: ensure .cpu() before .numpy() everywhere (idempotent)
    txt = txt.replace('.numpy()', '.cpu().numpy()').replace('.cpu().cpu().numpy()', '.cpu().numpy()')
    ESM_UTIL.write_text(txt)
    print('[OK] fair-esm util.py patched (biotite alias + .cpu().numpy())')

import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('transformers', transformers.__version__)
print('✅ environment ready')

cwd: /home/sagemaker-user/phageforge_clean


[OK] fair-esm util.py patched (biotite alias + .cpu().numpy())


torch 2.8.0 | cuda True | device NVIDIA A10G
transformers 4.57.6
✅ environment ready


## 2 — Common paths, sweep config, and seed selection

Seeds are chosen automatically to **span the distance axis** to the Enterobacter centroid (a few close, a few far), so the characterization in §5 covers a real range. `UOX38086.1` (the validated Klebsiella anchor) is always included.

In [2]:
import json, numpy as np, pandas as pd
from pathlib import Path
from phageforge.stage11_utils import (
    load_strict_dataset, load_strict_embeddings, compute_centroid, cosine_similarity,
)

# ---- fixed assets ----
STRICT_CSV       = ROOT / 'data/processed/rbp_dataset_eskapee_strict.csv'
STRICT_EMB_PT    = ROOT / 'data/processed/strict/esm2_embeddings.pt'
STRICT_EMB_IDX   = ROOT / 'data/processed/strict/esm2_embeddings_index.csv'
PREDICTOR_MODEL  = ROOT / 'results/broad/linear_probe/seed_42/model.joblib'
PREDICTOR_LABELS = ROOT / 'results/broad/linear_probe/seed_42/label_classes.json'
EMBEDDING_MODEL  = 'facebook/esm2_t33_650M_UR50D'
VALIDATOR        = 'scripts/08a_structural_fasttrack_validation.py'

# ---- sweep configuration (held constant across every seed) ----
TARGET_HOST   = 'Enterobacter'
SEED          = 42
N_SEEDS       = 6           # number of wild-type seeds in the sweep
MIN_LEN       = 400
MAX_LEN       = 850         # skip seeds longer than this (ESMFold VRAM safety on a 24 GB A10G)
ANCHOR_PID    = 'UOX38086.1'   # always include the validated Klebsiella anchor
# fixed search config:
MIN_MUT, MAX_MUT = 2, 8
ROUNDS, BEAM, PROPOSALS, SUBS = 6, 16, 10, 4
GATE_PLDDT    = 70.0

SWEEP_ROOT = ROOT / 'results/stage11_sweep'
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)

for p in [STRICT_CSV, STRICT_EMB_PT, STRICT_EMB_IDX, PREDICTOR_MODEL, PREDICTOR_LABELS]:
    assert p.exists(), f'Missing required asset: {p}'

# ---- distances to the Enterobacter centroid ----
strict_df = load_strict_dataset(STRICT_CSV)
emb, idx_df = load_strict_embeddings(STRICT_EMB_PT, STRICT_EMB_IDX)
idx_df = idx_df.reset_index(drop=True)

tgt_mask = (idx_df['host_genus'] == TARGET_HOST).to_numpy()
assert tgt_mask.sum() > 0, f'No {TARGET_HOST} rows in the strict embeddings index.'
tgt_centroid = compute_centroid(emb[tgt_mask])

# length lookup from the strict CSV
len_by_pid = {r.protein_id: len(str(r.aa_sequence)) for r in strict_df.itertuples()}
genus_by_pid = {r.protein_id: r.host_genus for r in strict_df.itertuples()}

rows = []
for i, r in idx_df.iterrows():
    pid = r['protein_id']; genus = r['host_genus']
    if genus == TARGET_HOST:                      # don't retarget Enterobacter onto itself
        continue
    L = len_by_pid.get(pid, 10**6)
    if L < MIN_LEN or L > MAX_LEN:
        continue
    cos = float(cosine_similarity(emb[i], tgt_centroid))
    rows.append({'protein_id': pid, 'source_host': genus, 'length': L,
                 'cos_to_target': cos, 'distance_to_target': 1.0 - cos})

cand = pd.DataFrame(rows).sort_values('distance_to_target').reset_index(drop=True)
print(f'{len(cand)} candidate seeds (<= {MAX_LEN} aa, non-{TARGET_HOST}).')

# pick N_SEEDS spanning the actual distance range evenly
min_d, max_d = cand['distance_to_target'].min(), cand['distance_to_target'].max()
target_distances = np.linspace(min_d, max_d, N_SEEDS)

picked_indices = []
for td in target_distances:
    # Find the closest available seed to this target distance
    available = cand.drop(picked_indices)
    if len(available) == 0: 
        break
    closest_idx = (available['distance_to_target'] - td).abs().idxmin()
    picked_indices.append(closest_idx)

picked = cand.loc[picked_indices].copy()

# always include the validated anchor
if ANCHOR_PID in set(cand['protein_id']) and ANCHOR_PID not in set(picked['protein_id']):
    picked = pd.concat([picked, cand[cand['protein_id'] == ANCHOR_PID]], ignore_index=True)
    
picked = picked.drop_duplicates('protein_id').sort_values('distance_to_target').reset_index(drop=True)

print('\nChosen sweep seeds (near -> far from Enterobacter):')
print(picked[['protein_id', 'source_host', 'length', 'cos_to_target', 'distance_to_target']].to_string(index=False))
SEEDS = picked.to_dict('records')


88 candidate seeds (<= 850 aa, non-Enterobacter).

Chosen sweep seeds (near -> far from Enterobacter):
protein_id    source_host  length  cos_to_target  distance_to_target
UAW09916.1  Acinetobacter     841       0.990340            0.009660
UOX38086.1     Klebsiella     806       0.982190            0.017810
WLY86866.1 Staphylococcus     641       0.980650            0.019350
QIA28516.1 Staphylococcus     481       0.972275            0.027725
QFR57578.1     Klebsiella     658       0.952647            0.047353
WWD14686.1     Klebsiella     659       0.951044            0.048956
WWD13915.1     Klebsiella     658       0.940915            0.059085


## 3 — Baseline Qualification control (the signature claim)

Folds a **shuffled** copy of the anchor sequence and shows the gate **rejects** it (pLDDT well below 70), while the intact wild-type **passes**. This empirically demonstrates that the gate discriminates a foldable chassis from a collapsed one — the core innovation of Stage 11. (Swap in the historical `round8_cand473` sequence here if you want to tie it to the exact Stage 06 artifact.)

In [3]:
import random
from phageforge.stage11_utils import esmfold_single_sequence, baseline_qualification_gate

CTRL_DIR = SWEEP_ROOT / 'gate_control'; CTRL_DIR.mkdir(parents=True, exist_ok=True)
anchor_seq = str(strict_df.loc[strict_df['protein_id'] == ANCHOR_PID, 'aa_sequence'].iloc[0])

# intact wild-type (positive control)
m_wt = esmfold_single_sequence(anchor_seq, CTRL_DIR / 'anchor_wt.pdb', device='cuda', chunk_size=128, num_recycles=1)
ok_wt, why_wt = baseline_qualification_gate(m_wt, GATE_PLDDT)

# shuffled / collapsed variant (negative control)
rng = random.Random(SEED)
shuffled = list(anchor_seq); rng.shuffle(shuffled); shuffled = ''.join(shuffled)
m_bad = esmfold_single_sequence(shuffled, CTRL_DIR / 'anchor_shuffled.pdb', device='cuda', chunk_size=128, num_recycles=1)
ok_bad, why_bad = baseline_qualification_gate(m_bad, GATE_PLDDT)

print(f'WILD-TYPE   mean pLDDT = {m_wt["mean_plddt"]:.2f}  -> gate {"PASS" if ok_wt else "REJECT"}')
print(f'SHUFFLED    mean pLDDT = {m_bad["mean_plddt"]:.2f}  -> gate {"PASS" if ok_bad else "REJECT"}')
json.dump({'wild_type_plddt': m_wt['mean_plddt'], 'wild_type_pass': bool(ok_wt),
           'shuffled_plddt': m_bad['mean_plddt'], 'shuffled_pass': bool(ok_bad),
           'threshold': GATE_PLDDT},
          open(CTRL_DIR / 'gate_control.json', 'w'), indent=2)
assert ok_wt and not ok_bad, 'Gate control unexpected — investigate before trusting the sweep.'
print('\n[OK] Gate discriminates foldable wild-type from collapsed sequence.')


2026-05-26 09:53:46.646225: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779789226.662993    1369 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779789226.669498    1369 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779789226.697219    1369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779789226.697239    1369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779789226.697242    1369 computation_placer.cc:177] computation placer alr

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


WILD-TYPE   mean pLDDT = 81.06  -> gate PASS
SHUFFLED    mean pLDDT = 20.35  -> gate REJECT

[OK] Gate discriminates foldable wild-type from collapsed sequence.


## 4 — The sweep: full 11a→11d per seed (top-3 validation only)

Default composite weights, fixed search config, `--reuse_cached_embeddings` for speed. A seed that fails the Baseline Qualification gate is recorded as `gate=REJECT` and skipped — a valid data point, not an error. Short run names keep paths well under the Windows 260-char limit.

In [3]:
import subprocess, sys, shutil, re, time

def run(cmd):
    return subprocess.run(cmd).returncode

records = []
for k, s in enumerate(SEEDS):
    pid, src_host, dist = s['protein_id'], s['source_host'], s['distance_to_target']
    tag = 'seed%02d_%s' % (k, re.sub(r'[^A-Za-z0-9]', '', str(pid)))
    rd  = SWEEP_ROOT / tag
    rd.mkdir(parents=True, exist_ok=True)
    print(f'\n========== [{k+1}/{len(SEEDS)}] {pid}  ({src_host}, dist={dist:.4f}) ==========', flush=True)
    rec = {'protein_id': pid, 'source_host': src_host, 'length': s['length'],
           'distance_to_target': dist, 'run_dir': str(rd),
           'gate': 'ERROR', 'seed_plddt': np.nan, 'top3_pass': 0, 'top3_n': 0,
           'best_cand_plddt': np.nan, 'best_rmsd': np.nan, 'best_target_prob': np.nan}
    try:
        t0 = time.time()
        # ---- 11a: context + ESMFold seed + Baseline Qualification gate ----
        rc = run([sys.executable, 'scripts/11a_prepare_stage11_context.py',
            '--strict_csv', str(STRICT_CSV), '--seed_protein_id', pid,
            '--source_host', src_host, '--target_host', TARGET_HOST,
            '--out_dir', str(rd), '--run_name', tag,
            '--strict_embeddings', str(STRICT_EMB_PT), '--strict_embeddings_index', str(STRICT_EMB_IDX),
            '--predictor_model', str(PREDICTOR_MODEL), '--predictor_label_classes', str(PREDICTOR_LABELS),
            '--embedding_model', EMBEDDING_MODEL, '--reuse_cached_embeddings',
            '--esmfold_device', 'cuda', '--esmfold_chunk_size', '128', '--esmfold_num_recycles', '1',
            '--min_seed_plddt', str(GATE_PLDDT),
            '--min_mutations', str(MIN_MUT), '--max_mutations', str(MAX_MUT),
            '--seed', str(SEED)])
        ctx = rd / 'context' / 'stage11_context.json'
        if rc == 2:
            rec['gate'] = 'REJECT'
            bq = rd / 'context' / 'baseline_qualification.json'
            if bq.exists(): rec['seed_plddt'] = json.load(open(bq)).get('seed_mean_plddt', np.nan)
            print('  -> gate REJECT; skipping search.'); records.append(rec); continue
        assert rc == 0 and ctx.exists(), f'11a failed (rc={rc})'
        rec['gate'] = 'PASS'
        rec['seed_plddt'] = json.load(open(ctx))['baseline_qualification']['seed_mean_plddt']

        # ---- 11b: inverse-folding beam search (default weights) ----
        search_csv = rd / 'search' / 'stage11_search_candidates.csv'
        rc = run([sys.executable, 'scripts/11b_run_inverse_folding_beam_search.py',
            '--stage11_context_json', str(ctx),
            '--predictor_model', str(PREDICTOR_MODEL), '--label_classes_json', str(PREDICTOR_LABELS),
            '--out_csv', str(search_csv), '--out_json', str(rd / 'search' / 'stage11_search_summary.json'),
            '--embedding_model', EMBEDDING_MODEL, '--if_device', 'cuda', '--if_chain_id', 'A',
            '--rounds', str(ROUNDS), '--beam_width', str(BEAM),
            '--proposals_per_parent', str(PROPOSALS), '--substitutions_per_position', str(SUBS),
            '--batch_size', '4', '--seed', str(SEED)])
        assert rc == 0, f'11b failed (rc={rc})'

        # ---- 11c: two-pass diversity prefilter ----
        top3 = rd / 'prefilter' / 'stage11_top3.csv'
        rc = run([sys.executable, 'scripts/11c_prefilter_stage11_candidates.py',
            '--stage11_context_json', str(ctx), '--search_csv', str(search_csv),
            '--out_topk_csv', str(rd / 'prefilter' / 'stage11_top10.csv'),
            '--out_topk_final_csv', str(top3),
            '--out_json', str(rd / 'prefilter' / 'stage11_prefilter_summary.json'),
            '--top_k', '10', '--top_k_final', '3', '--embedding_model', EMBEDDING_MODEL,
            '--batch_size', '4', '--seed', str(SEED)])
        assert rc == 0, f'11c failed (rc={rc})'
        try: rec['best_target_prob'] = float(pd.read_csv(top3)['target_probability'].max())
        except Exception: pass

        # ---- 11d: structural validation of the top-3 (08a oracle) ----
        vdir = rd / 'validation_top3'
        rc = run([sys.executable, 'scripts/11d_validate_stage11_candidates.py',
            '--validated_csv', str(top3), '--ranked_csv', str(top3), '--context_json', str(ctx),
            '--validator_script', VALIDATOR, '--out_dir', str(vdir),
            '--out_json', str(vdir / 'stage11_top3_launch.json'),
            '--top_k', '3', '--device', 'cuda', '--chunk_size', '128', '--num_recycles', '1'])
        assert rc == 0, f'11d failed (rc={rc})'
        vsum = vdir / 'stage08_structural_fasttrack_summary.csv'
        if vsum.exists():
            vdf = pd.read_csv(vsum)
            rec['top3_n']   = int(len(vdf))
            rec['top3_pass'] = int(vdf['stage08_pass'].sum())
            rec['best_cand_plddt'] = float(vdf['esmfold_mean_plddt'].max())
            rec['best_rmsd'] = float(vdf['rmsd_to_selected_seed'].min())
        print(f'  -> PASS {rec["top3_pass"]}/{rec["top3_n"]} | seed pLDDT {rec["seed_plddt"]:.1f} '
              f'| best cand pLDDT {rec["best_cand_plddt"]:.1f} | best RMSD {rec["best_rmsd"]:.2f} '
              f'| best target_p {rec["best_target_prob"]:.3f} | {time.time()-t0:.0f}s')
    except Exception as e:
        rec['gate'] = rec['gate'] if rec['gate'] != 'ERROR' else 'ERROR'
        print('  -> ERROR:', repr(e))
    records.append(rec)

sweep_df = pd.DataFrame(records)
sweep_df.to_csv(SWEEP_ROOT / 'sweep_summary.csv', index=False)
print('\n[OK] wrote', SWEEP_ROOT / 'sweep_summary.csv')
sweep_df



========== [1/7] UAW09916.1  (Acinetobacter, dist=0.0097) ==========


[INFO] Stage 11a — seed=UAW09916.1 (Acinetobacter → Enterobacter), length=841 AA, run_name=seed00_UAW099161
[INFO] Folding seed with ESMFold (device=cuda, chunk=128, recycles=1)…


2026-05-26 10:42:30.022964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779792150.038888    2958 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779792150.044472    2958 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779792150.058080    2958 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779792150.058111    2958 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779792150.058114    2958 computation_placer.cc:177] computation placer alr

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GATE] Baseline Qualification PASSED: Seed mean pLDDT=74.029 passes threshold 70.000.
[INFO] Loading ESM-2 backbone 'facebook/esm2_t33_650M_UR50D' for centroid math…


...

2026-05-26 11:19:46.506476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779794386.522400    3902 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779794386.527958    3902 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779794386.541471    3902 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779794386.541501    3902 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779794386.541504    3902 computation_placer.cc:177] computation placer alr

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[GATE] Baseline Qualification FAILED: Seed mean pLDDT=22.296 is below the Baseline Qualification threshold (70.000). Pick a different seed and rerun.
[GATE] Wrote gate evidence to: /home/sagemaker-user/phageforge_clean/results/stage11_sweep/seed06_WWD139151/context/baseline_qualification.json


  -> gate REJECT; skipping search.

[OK] wrote /home/sagemaker-user/phageforge_clean/results/stage11_sweep/sweep_summary.csv


,protein_id,source_host,length,distance_to_target,run_dir,gate,seed_plddt,top3_pass,top3_n,best_cand_plddt,best_rmsd,best_target_prob
0,UAW09916.1,Acinetobacter,841,0.009660,/home/sagemaker-user/phageforge_clean/results/...,PASS,74.028537,3,3,72.941736,1.277668,0.098854
1,UOX38086.1,Klebsiella,806,0.017810,/home/sagemaker-user/phageforge_clean/results/...,PASS,81.057072,3,3,78.754342,1.271272,0.220110
2,WLY86866.1,Staphylococcus,641,0.019350,/home/sagemaker-user/phageforge_clean/results/...,REJECT,69.382215,0,0,NaN,NaN,NaN
3,QIA28516.1,Staphylococcus,481,0.027725,/home/sagemaker-user/phageforge_clean/results/...,PASS,88.149688,3,3,88.706861,0.188475,0.174904
4,QFR57578.1,Klebsiella,658,0.047353,/home/sagemaker-user/phageforge_clean/results/...,REJECT,23.919453,0,0,NaN,NaN,NaN
5,WWD14686.1,Klebsiella,659,0.048956,/home/sagemaker-user/phageforge_clean/results/...,REJECT,22.540212,0,0,NaN,NaN,NaN
6,WWD13915.1,Klebsiella,658,0.059085,/home/sagemaker-user/phageforge_clean/results/...,REJECT,22.296353,0,0,NaN,NaN,NaN


## 5 — Summary table + figures

Produces the *money table* and two portfolio figures: (1) per-seed structural pass rate and seed pLDDT, (2) seed-to-target distance vs predicted target-host probability.

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = SWEEP_ROOT / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
df = sweep_df.copy()
df['label'] = df['protein_id'].astype(str) + '\n(' + df['source_host'].astype(str) + ')'
passed = df[df['gate'] == 'PASS'].copy()

# ----- headline numbers -----
n_pass_gate = int((df['gate'] == 'PASS').sum())
n_total = int(len(df))
total_pass = int(passed['top3_pass'].sum()); total_n = int(passed['top3_n'].sum())
print(f'Seeds qualified by gate: {n_pass_gate}/{n_total}')
print(f'Aggregate top-3 structural pass rate (qualified seeds): {total_pass}/{total_n} '
      f'= {100.0*total_pass/max(total_n,1):.0f}%')
print(f'Mean seed pLDDT (qualified): {passed["seed_plddt"].mean():.1f}')

# ----- Figure 1: pass rate + seed pLDDT per seed -----
fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(df))
rate = np.where(df['top3_n'] > 0, 100.0 * df['top3_pass'] / df['top3_n'].replace(0, np.nan), 0.0)
bars = ax1.bar(x, rate, color=['#3b7dd8' if g == 'PASS' else '#c0c0c0' for g in df['gate']])
ax1.set_ylabel('Top-3 structural pass rate (%)'); ax1.set_ylim(0, 105)
ax1.set_xticks(x); ax1.set_xticklabels(df['label'], rotation=45, ha='right', fontsize=8)
ax2 = ax1.twinx()
ax2.plot(x, df['seed_plddt'], 'o-', color='#d8632b', label='seed pLDDT')
ax2.axhline(GATE_PLDDT, ls='--', color='#999', lw=1); ax2.set_ylabel('seed mean pLDDT'); ax2.set_ylim(0, 100)
ax2.text(len(df)-0.5, GATE_PLDDT+1, 'gate 70', color='#777', fontsize=8, ha='right')
ax1.set_title('Stage 11 multi-seed sweep — structural pass rate and seed foldability')
fig.tight_layout(); fig.savefig(FIG_DIR / 'sweep_passrate.png', dpi=160); plt.close(fig)

# ----- Figure 2: distance-to-target vs predicted target probability -----
fig, ax = plt.subplots(figsize=(7, 5))
sc = passed.dropna(subset=['best_target_prob'])
ax.scatter(sc['distance_to_target'], sc['best_target_prob'], s=70, c='#3b7dd8', edgecolor='k', zorder=3)
for _, r in sc.iterrows():
    ax.annotate(str(r['protein_id']), (r['distance_to_target'], r['best_target_prob']),
                fontsize=7, xytext=(4, 4), textcoords='offset points')
if len(sc) >= 2:
    m, b = np.polyfit(sc['distance_to_target'], sc['best_target_prob'], 1)
    xs = np.linspace(sc['distance_to_target'].min(), sc['distance_to_target'].max(), 50)
    ax.plot(xs, m*xs + b, '--', color='#d8632b', lw=1.5, zorder=2)
    rcorr = np.corrcoef(sc['distance_to_target'], sc['best_target_prob'])[0, 1]
    ax.set_title(f'Seed-distance vs predicted target probability  (r = {rcorr:.2f})')
else:
    ax.set_title('Seed-distance vs predicted target probability')
ax.set_xlabel(f'embedding distance to {TARGET_HOST} centroid (1 - cosine)')
ax.set_ylabel('best top-3 target_probability')
fig.tight_layout(); fig.savefig(FIG_DIR / 'distance_vs_targetprob.png', dpi=160); plt.close(fig)

print('[OK] figures ->', FIG_DIR)
disp_cols = ['protein_id','source_host','length','distance_to_target','gate',
             'seed_plddt','top3_pass','top3_n','best_cand_plddt','best_rmsd','best_target_prob']
df[disp_cols]


Seeds qualified by gate: 3/7
Aggregate top-3 structural pass rate (qualified seeds): 9/9 = 100%
Mean seed pLDDT (qualified): 81.1


[OK] figures -> /home/sagemaker-user/phageforge_clean/results/stage11_sweep/figures


,protein_id,source_host,length,distance_to_target,gate,seed_plddt,top3_pass,top3_n,best_cand_plddt,best_rmsd,best_target_prob
0,UAW09916.1,Acinetobacter,841,0.009660,PASS,74.028537,3,3,72.941736,1.277668,0.098854
1,UOX38086.1,Klebsiella,806,0.017810,PASS,81.057072,3,3,78.754342,1.271272,0.220110
2,WLY86866.1,Staphylococcus,641,0.019350,REJECT,69.382215,0,0,NaN,NaN,NaN
3,QIA28516.1,Staphylococcus,481,0.027725,PASS,88.149688,3,3,88.706861,0.188475,0.174904
4,QFR57578.1,Klebsiella,658,0.047353,REJECT,23.919453,0,0,NaN,NaN,NaN
5,WWD14686.1,Klebsiella,659,0.048956,REJECT,22.540212,0,0,NaN,NaN,NaN
6,WWD13915.1,Klebsiella,658,0.059085,REJECT,22.296353,0,0,NaN,NaN,NaN


## 6 — Package the sweep for download

In [5]:
import tarfile
archive = ROOT / 'stage11_sweep_results.tar.gz'
with tarfile.open(archive, 'w:gz') as tar:
    tar.add(SWEEP_ROOT / 'sweep_summary.csv', arcname='sweep_summary.csv')
    tar.add(SWEEP_ROOT / 'figures', arcname='figures')
    tar.add(SWEEP_ROOT / 'gate_control', arcname='gate_control')
    for s in SEEDS:
        tag = [p.name for p in SWEEP_ROOT.iterdir() if p.is_dir() and p.name.endswith(re.sub(r'[^A-Za-z0-9]','',str(s['protein_id'])))]
        for t in tag:
            d = SWEEP_ROOT / t
            for sub in ['context','prefilter','report','validation_top3']:
                if (d/sub).exists(): tar.add(d/sub, arcname=f'{t}/{sub}')
print('Archive:', archive, f'({archive.stat().st_size/1e6:.1f} MB)')
print('Right-click in the JupyterLab file pane -> Download.')


Archive: /home/sagemaker-user/phageforge_clean/stage11_sweep_results.tar.gz (2.4 MB)
Right-click in the JupyterLab file pane -> Download.
